# Лабораторная работа №2

## Задание 1. Сформировать отчёт с информацией о 10 наиболее популярных языках программирования по итогам года за период с 2010 по 2020 годы.

In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import year, to_timestamp
import pyspark.sql.functions as F
import xml.etree.ElementTree as ET

spark = SparkSession.builder \
    .appName("Lab2") \
    .getOrCreate()
print("Spark session created: ")
spark

Spark session created: 


In [13]:
tree = ET.parse("posts_sample.xml")
root = tree.getroot()
data = []

# Парсим xml файл
for row in root.findall('row'):
  # Получаем нужные поля
    date = row.attrib.get('CreationDate')
    tags = row.attrib.get('Tags')
    if tags is not None:
        # Т.к. в тэге поста может много языков - удаляем лишние символы и разбиваем на массив тэгов
        tags = tags.replace('<', '').replace('>', ' ').strip().split()
        for tag in tags:
            data.append((date, tag))

data

[('2008-07-31T21:42:52.667', 'c#'),
 ('2008-07-31T21:42:52.667', 'floating-point'),
 ('2008-07-31T21:42:52.667', 'type-conversion'),
 ('2008-07-31T21:42:52.667', 'double'),
 ('2008-07-31T21:42:52.667', 'decimal'),
 ('2008-07-31T22:08:08.620', 'html'),
 ('2008-07-31T22:08:08.620', 'css'),
 ('2008-07-31T22:08:08.620', 'internet-explorer-7'),
 ('2008-07-31T23:40:59.743', 'c#'),
 ('2008-07-31T23:40:59.743', '.net'),
 ('2008-07-31T23:40:59.743', 'datetime'),
 ('2008-07-31T23:55:37.967', 'c#'),
 ('2008-07-31T23:55:37.967', 'datetime'),
 ('2008-07-31T23:55:37.967', 'time'),
 ('2008-07-31T23:55:37.967', 'datediff'),
 ('2008-07-31T23:55:37.967', 'relative-time-span'),
 ('2008-08-01T00:42:38.903', 'html'),
 ('2008-08-01T00:42:38.903', 'browser'),
 ('2008-08-01T00:42:38.903', 'timezone'),
 ('2008-08-01T00:42:38.903', 'user-agent'),
 ('2008-08-01T00:42:38.903', 'timezone-offset'),
 ('2008-08-01T00:59:11.177', '.net'),
 ('2008-08-01T00:59:11.177', 'math'),
 ('2010-09-22T10:33:21.790', 'c++'),
 ('20

In [14]:
# Создание схемы DataFrame
schema = StructType([
    StructField("CreationDate", StringType(), True),
    StructField("Tag", StringType(), True)
])

# Создание DataFrame
xml_df = spark.createDataFrame(data, schema=schema)

# Извлекаем год из доты
xml_df = xml_df.withColumn("Year", year(to_timestamp(xml_df["CreationDate"], "yyyy-MM-dd'T'HH:mm:ss.SSS")))

# Дропаем ненужный столбец
xml_df = xml_df.drop("CreationDate")

xml_df.show()


+-------------------+----+
|                Tag|Year|
+-------------------+----+
|                 c#|2008|
|     floating-point|2008|
|    type-conversion|2008|
|             double|2008|
|            decimal|2008|
|               html|2008|
|                css|2008|
|internet-explorer-7|2008|
|                 c#|2008|
|               .net|2008|
|           datetime|2008|
|                 c#|2008|
|           datetime|2008|
|               time|2008|
|           datediff|2008|
| relative-time-span|2008|
|               html|2008|
|            browser|2008|
|           timezone|2008|
|         user-agent|2008|
+-------------------+----+
only showing top 20 rows


In [10]:
schema = StructType([
    StructField("Tag", StringType(), True),
    StructField("wikipedia_url", StringType(), True)
])

# Читаем CSV файл в DataFrame
csv_df = spark.read.csv("programming-languages.csv", schema=schema)

# Добавляем индекс к DataFrame
csv_df = csv_df.rdd.zipWithIndex().toDF(["data", "index"])

# Извлекаем данные обратно в DataFrame
csv_df = csv_df.filter(csv_df.index != 0).select("data.*")

from pyspark.sql.functions import lower

csv_df = csv_df.withColumn("Tag", lower(csv_df["Tag"]))

csv_df.show()

+------------+--------------------+
|         Tag|       wikipedia_url|
+------------+--------------------+
|     a# .net|https://en.wikipe...|
|  a# (axiom)|https://en.wikipe...|
|  a-0 system|https://en.wikipe...|
|          a+|https://en.wikipe...|
|         a++|https://en.wikipe...|
|        abap|https://en.wikipe...|
|         abc|https://en.wikipe...|
|   abc algol|https://en.wikipe...|
|       abset|https://en.wikipe...|
|       absys|https://en.wikipe...|
|         acc|https://en.wikipe...|
|      accent|https://en.wikipe...|
|    ace dasl|https://en.wikipe...|
|        acl2|https://en.wikipe...|
|     act-iii|https://en.wikipe...|
|     action!|https://en.wikipe...|
|actionscript|https://en.wikipe...|
|         ada|https://en.wikipe...|
|     adenine|https://en.wikipe...|
|        agda|https://en.wikipe...|
+------------+--------------------+
only showing top 20 rows


In [15]:
# Джоиним таблицы, чтобы убрать теги без языков программирования
df = xml_df.join(csv_df, on="Tag", how="inner")
df = df.drop("wikipedia_url")

df.show()

+----+----+
| Tag|Year|
+----+----+
|dart|2019|
|dart|2019|
|dart|2013|
|dart|2018|
|dart|2013|
|dart|2015|
|dart|2019|
|dart|2019|
|dart|2018|
|dart|2019|
|dart|2019|
|dart|2019|
|dart|2018|
|dart|2019|
|dart|2019|
|dart|2013|
|curl|2018|
|curl|2014|
|curl|2012|
|curl|2018|
+----+----+
only showing top 20 rows


In [17]:
from pyspark.sql import Window

# Группируем по году и языку, считаем количество упоминаний
language_counts = df.groupBy("Tag", "Year").count()

# Определяем окно для ранжирования
window_spec = Window.partitionBy("Year").orderBy(F.desc("count"))

# Добавляем ранг и фильтруем топ-10 языков для каждого года
top_languages = language_counts.withColumn("rank", F.row_number().over(window_spec)) \
    .filter(F.col("rank") <= 10) \
    .drop("rank")

top_languages = top_languages.orderBy("Year", F.desc("count"))



## Задание 2. Получившийся отчёт сохранить в формате Apache Parquet.

In [18]:

# Сохраняем результат в формате Parquet
top_languages.write.mode("overwrite").parquet("/content/top_10_languages_by_year.parquet")